# Ejercicio 10: Re-ranking

### **Objetivo**: *Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.*

## Leandro Bravo

## Parte 1. Preparación del corpus
* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
import os
import sys
from pathlib import Path
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score
import lightgbm as lgb
from sklearn.model_selection import train_test_split

from beir.retrieval.evaluation import EvaluateRetrieval

current_dir = Path.cwd()
ir_root = current_dir.parent
rerank_dir = ir_root / '06Ev'

if str(rerank_dir) not in sys.path:
    sys.path.insert(0, str(rerank_dir))
from bm25_model import build_bm25_index, bm25_rank, tokenize

C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\beir\util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../10rerank/data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

'../10rerank/data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)
* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [8]:
from beir.retrieval.evaluation import EvaluateRetrieval
def run_beir_evaluation(df_queries, df_corpus, ground_truth, index, k_values=[10]):    
    # 1. Asegurar el formato de BEIR para qrels (llaves y valores como string)
    qrels = {}
    for q_id, docs in ground_truth.items():
        qrels[str(q_id)] = {str(d_id): int(rel) for d_id, rel in docs.items()}

    results = {}
    
    # Iteramos fila por fila usando los IDs reales de tus consultas
    for _, q_row in df_queries.iterrows():
        q_id = str(q_row["query_id"])
        query_text = q_row["query"]
        
        results[q_id] = {}
        
        # Ejecutar BM25 con el texto de los documentos originales
        df_rank = bm25_rank(query_text, df_corpus["text"].tolist(), index)
        
        # Mapear el 'Document_ID' posicional que devuelve BM25 al ID real de tu corpus
        for _, r_row in df_rank.iterrows():
            pos_idx = int(r_row['Document_ID'])
            
            # Obtener el ID real correspondiente a esa posición en el DataFrame
            real_doc_id = str(df_corpus.iloc[pos_idx]["doc_id"])
            
            results[q_id][real_doc_id] = float(r_row['BM25_Score'])

    # 3. Inicializar el evaluador de BEIR
    evaluator = EvaluateRetrieval(retriever=None, k_values=k_values)
    
    # 4. Calcular métricas (Ahora len(scores) será > 0 al haber match de IDs)
    ndcg, _map, recall, precision = evaluator.evaluate(qrels, results, k_values)
    
    return ndcg, recall, precision

In [9]:
# Uso de la función
run_beir_evaluation(df_queries=df_queries,
                    df_corpus=df_corpus,
                    ground_truth=qrels, 
                    index=build_bm25_index(df_corpus["text"].tolist()),
                    k_values=[10])

({'NDCG@10': 0.64835}, {'Recall@10': 0.76717}, {'P@10': 0.08467})

## Parte 3. Implementación del re-ranking cross-encoder
* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [10]:
from sentence_transformers import CrossEncoder

def run_cross_encoder_reranking(df_queries, df_corpus, bm25_index, top_k_baseline=20, top_n_final=10):
    """Ejecuta el re-ranking con Cross-Encoder sobre el top_k de BM25"""
    
    # 1. Inicializar el modelo Cross-Encoder estándar para Text Retrieval
    print("Cargando modelo Cross-Encoder...")
    model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    corpus_texts = df_corpus["text"].tolist()
    all_comparisons = []

    # 2. Iterar sobre cada consulta
    for _, q_row in df_queries.iterrows():
        q_id = q_row["query_id"]
        query_text = q_row["query"]

        # FASE 1: Retrieval Inicial (BM25)
        df_bm25 = bm25_rank(query_text, corpus_texts, bm25_index)
        
        # Mapear el ID posicional al ID real de tu corpus
        df_bm25["real_doc_id"] = df_bm25["Document_ID"].apply(lambda idx: df_corpus.iloc[idx]["doc_id"])
        
        # Guardamos el rango inicial de BM25 (1, 2, 3...)
        df_bm25["Rank_BM25"] = df_bm25.index + 1
        
        # Seleccionar el top_k de candidatos para el re-ranking (ej. top 20)
        candidates = df_bm25.head(top_k_baseline).copy()

        # FASE 2: Re-ranking (Cross-Encoder)
        pairs = [[query_text, corpus_texts[int(doc_idx)]] for doc_idx in candidates["Document_ID"]]
        
        # Calcular los nuevos scores de similitud semántica
        ce_scores = model.predict(pairs)
        candidates["CE_Score"] = ce_scores

        # Ordenar por el nuevo score del Cross-Encoder
        candidates = candidates.sort_values(by="CE_Score", ascending=False).reset_index(drop=True)
        candidates["Rank_CE"] = candidates.index + 1

        # FASE 3: Identificar Cambios de Posición en el Top N (ej. Top 10)
        # Documentos que terminaron en el Top N final tras el Re-ranking
        top_final_ce = candidates.head(top_n_final).copy()

        for _, row in top_final_ce.iterrows():
            doc_id = row["real_doc_id"]
            pos_bm25 = int(row["Rank_BM25"])
            pos_ce = int(row["Rank_CE"])
            shift = pos_bm25 - pos_ce  # Positivo si subió (ej: de 12 a 3 -> +9)

            if pos_bm25 > top_n_final:
                status = f"Entró al Top {top_n_final} (Estaba en posición {pos_bm25} de BM25)"
            elif shift > 0:
                status = f"Subió {shift} posiciones"
            elif shift < 0:
                status = f"Bajó {abs(shift)} posiciones"
            else:
                status = "Mantuvo su posición"

            all_comparisons.append({
                "Query_ID": q_id,
                "Query": query_text,
                "Doc_ID": doc_id,
                "Rank_BM25": pos_bm25,
                "Rank_CE": pos_ce,
                "Cambio": status,
                "Texto_Doc": corpus_texts[int(row["Document_ID"])][:80] + "..."
            })

    return pd.DataFrame(all_comparisons)

In [11]:
run_cross_encoder_reranking(df_queries=df_queries,
                            df_corpus=df_corpus,
                            bm25_index=build_bm25_index(df_corpus["text"].tolist()),
                            top_k_baseline=10,
                            top_n_final=10)

Cargando modelo Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,Query_ID,Query,Doc_ID,Rank_BM25,Rank_CE,Cambio,Texto_Doc
0,1,0-dimensional biomaterials show inductive prop...,43385013,3,1,Subió 2 posiciones,It has been proposed that epithelial-mesenchym...
1,1,0-dimensional biomaterials show inductive prop...,10906636,6,2,Subió 4 posiciones,US28 is one of four 7 transmembrane (7TM) chem...
2,1,0-dimensional biomaterials show inductive prop...,10608397,1,3,Bajó 2 posiciones,BACKGROUND Paralysis or amputation of an arm r...
3,1,0-dimensional biomaterials show inductive prop...,27049238,4,4,Mantuvo su posición,Red blood cells are known to change shape in r...
4,1,0-dimensional biomaterials show inductive prop...,3770726,10,5,Subió 5 posiciones,BACKGROUND Microfluidic platforms for quantita...
...,...,...,...,...,...,...,...
2995,1395,p16INK4A accumulation is linked to an abnorma...,4387484,8,6,Subió 2 posiciones,The Kaposi's sarcoma-associated herpesvirus (K...
2996,1395,p16INK4A accumulation is linked to an abnorma...,7482674,10,7,Subió 3 posiciones,Pelizaeus-Merzbacher disease (PMD) is a form o...
2997,1395,p16INK4A accumulation is linked to an abnorma...,40987633,6,8,Bajó 2 posiciones,Charged multivesicular body protein 2B (CHMP2B...
2998,1395,p16INK4A accumulation is linked to an abnorma...,10698739,4,9,Bajó 5 posiciones,Loss of Omi/HtrA2 function leads to nerve cell...


## Parte 4. Implementación del re-ranking LTR
* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [12]:
def extract_features(query_text, doc_text, bm25_score, index):
    q_tokens = tokenize(query_text)
    d_tokens = tokenize(doc_text)
    
    # Feature 1: Score de BM25 (es nuestra señal base fuerte)
    f_bm25 = bm25_score
    
    # Feature 2: Coincidencia exacta de términos (Jaccard Similarity)
    set_q = set(q_tokens)
    set_d = set(d_tokens)
    intersection = set_q.intersection(set_d)
    union = set_q.union(set_d)
    f_jaccard = len(intersection) / len(union) if union else 0.0
    
    # Feature 3: Proporción de palabras de la query que están en el documento
    f_query_coverage = len(intersection) / len(set_q) if set_q else 0.0
    
    # Feature 4: Longitud del documento
    f_doc_len = len(d_tokens)
    
    return [f_bm25, f_jaccard, f_query_coverage, f_doc_len]

In [13]:
def prepare_ltr_dataset(df_queries, df_corpus, ground_truth, bm25_index, top_k_baseline=20):
    X = []
    y = []
    groups = []
    
    corpus_texts = df_corpus["text"].tolist()

    for _, q_row in df_queries.iterrows():
        q_id = q_row["query_id"]
        query_text = q_row["query"]
        
        # Obtener candidatos de BM25
        df_bm25 = bm25_rank(query_text, corpus_texts, bm25_index)
        df_bm25["real_doc_id"] = df_bm25["Document_ID"].apply(lambda idx: df_corpus.iloc[idx]["doc_id"])
        candidates = df_bm25.head(top_k_baseline)
        
        query_relevances = ground_truth.get(query_text, {})
        count_candidates = 0
        
        for _, row in candidates.iterrows():
            doc_idx = int(row["Document_ID"])
            real_doc_id = row["real_doc_id"]
            bm25_score = row["BM25_Score"]
            
            # Extraer características
            features = extract_features(query_text, corpus_texts[doc_idx], bm25_score, bm25_index)
            X.append(features)
            
            # Obtener etiqueta de relevancia real (0 si no está en ground_truth)
            # Buscamos tanto por el índice posicional como por el id real según cómo estructuraste tu qrels
            relevance = query_relevances.get(doc_idx, query_relevances.get(real_doc_id, 0))
            y.append(relevance)
            
            count_candidates += 1
            
        if count_candidates > 0:
            groups.append(count_candidates)
            
    return np.array(X), np.array(y), groups

In [14]:
def run_ltr_reranking(df_queries, df_corpus, bm25_index, ground_truth, top_k_baseline=20, top_n_final=10):
    X, y, groups = prepare_ltr_dataset(df_queries, df_corpus, ground_truth, bm25_index, top_k_baseline)
    
    if len(groups) == 0:
        print("Error: No se pudieron mapear grupos de entrenamiento. Revisa los IDs de tu ground_truth.")
        return pd.DataFrame()

    # 2. Configurar y entrenar el modelo LightGBM Ranker
    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[top_n_final],
        n_estimators=50,
        learning_rate=0.1,
        verbose=-1
    )
    
    print("Entrenando el modelo LTR (LambdaRANK)...")
    ranker.fit(X, y, group=groups)
    
    # 3. Aplicar Re-ranking e identificar variaciones en el orden
    corpus_texts = df_corpus["text"].tolist()
    all_comparisons = []
    
    for _, q_row in df_queries.iterrows():
        q_id = q_row["query_id"]
        query_text = q_row["query"]
        
        df_bm25 = bm25_rank(query_text, corpus_texts, bm25_index)
        df_bm25["real_doc_id"] = df_bm25["Document_ID"].apply(lambda idx: df_corpus.iloc[idx]["doc_id"])
        df_bm25["Rank_BM25"] = df_bm25.index + 1
        
        candidates = df_bm25.head(top_k_baseline).copy()
        
        # Construir features para los candidatos de esta query específica
        X_cand = []
        for _, row in candidates.iterrows():
            feats = extract_features(query_text, corpus_texts[int(row["Document_ID"])], row["BM25_Score"], bm25_index)
            X_cand.append(feats)
            
        # Predecir puntuaciones de ordenamiento LTR
        ltr_scores = ranker.predict(np.array(X_cand))
        candidates["LTR_Score"] = ltr_scores
        
        # Ordenar por el nuevo criterio aprendido por LTR
        candidates = candidates.sort_values(by="LTR_Score", ascending=False).reset_index(drop=True)
        candidates["Rank_LTR"] = candidates.index + 1
        
        # Filtrar el Top N definitivo para evaluar desplazamientos
        top_final_ltr = candidates.head(top_n_final)
        
        for _, row in top_final_ltr.iterrows():
            pos_bm25 = int(row["Rank_BM25"])
            pos_ltr = int(row["Rank_LTR"])
            shift = pos_bm25 - pos_ltr
            
            if pos_bm25 > top_n_final:
                status = f"Entró al Top {top_n_final} (Posición original BM25: {pos_bm25})"
            elif shift > 0:
                status = f"Subió {shift} posiciones"
            elif shift < 0:
                status = f"Bajó {abs(shift)} posiciones"
            else:
                status = "Mantuvo su posición"
                
            all_comparisons.append({
                "Query_ID": q_id,
                "Query": query_text,
                "Doc_ID": row["real_doc_id"],
                "Rank_BM25": pos_bm25,
                "Rank_LTR": pos_ltr,
                "Cambio": status
            })
            
    return pd.DataFrame(all_comparisons)

In [15]:
run_ltr_reranking(df_queries=df_queries,
                  df_corpus=df_corpus,
                  bm25_index=build_bm25_index(df_corpus["text"].tolist()),
                  ground_truth=qrels,
                  top_k_baseline=20,
                  top_n_final=10)

Entrenando el modelo LTR (LambdaRANK)...


C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
C:\Users\leand\AppData\Local\Packages

,Query_ID,Query,Doc_ID,Rank_BM25,Rank_LTR,Cambio
0,1,0-dimensional biomaterials show inductive prop...,10608397,1,1,Mantuvo su posición
1,1,0-dimensional biomaterials show inductive prop...,40212412,2,2,Mantuvo su posición
2,1,0-dimensional biomaterials show inductive prop...,27772649,19,3,Entró al Top 10 (Posición original BM25: 19)
3,1,0-dimensional biomaterials show inductive prop...,42240424,18,4,Entró al Top 10 (Posición original BM25: 18)
4,1,0-dimensional biomaterials show inductive prop...,825728,17,5,Entró al Top 10 (Posición original BM25: 17)
...,...,...,...,...,...,...
2995,1395,p16INK4A accumulation is linked to an abnorma...,1944452,16,6,Entró al Top 10 (Posición original BM25: 16)
2996,1395,p16INK4A accumulation is linked to an abnorma...,3419802,15,7,Entró al Top 10 (Posición original BM25: 15)
2997,1395,p16INK4A accumulation is linked to an abnorma...,26079071,14,8,Entró al Top 10 (Posición original BM25: 14)
2998,1395,p16INK4A accumulation is linked to an abnorma...,31616203,13,9,Entró al Top 10 (Posición original BM25: 13)


## Parte 5. Evaluación post re-ranking
Calcular métricas:

* nDCG@10
* MAP
* Recall@10

In [16]:
def evaluate_pipeline_with_beir(qrels_dict, baseline_results, ce_results, ltr_results, k_values=[10]):
    
    k_str = f"NDCG@{k_values[0]}"  
    k_num = k_values[0]
    
    summary_rows = []
    
    try:
        ndcg_b, recall_b, precision_b = baseline_results
        summary_rows.append({
            "Sistema": "Baseline (BM25)",
            "nDCG@10": ndcg_b.get(f"NDCG@{k_num}", 0.0),
            "MAP@10": 0.0,  # Cambiar por el valor real si tu tupla lo incluye
            "Recall@10": recall_b.get(f"Recall@{k_num}", 0.0)
        })
    except Exception as e:
        print(f"Nota: No se pudo desempaquetar la tupla del baseline automáticamente: {e}")

    rerankers = {
        "Re-ranking (Cross-Encoder)": ce_results,
        "Re-ranking (LTR - LambdaRANK)": ltr_results
    }
    
    for name, df in rerankers.items():
        if isinstance(df, pd.DataFrame) and not df.empty:
            results_dict = {}
            rank_col = "Rank_CE" if "Rank_CE" in df.columns else "Rank_LTR"
            
            for _, row in df.iterrows():
                q_id = str(row["Query_ID"])
                d_id = str(row["Doc_ID"])
                score = 1 / int(row[rank_col])
                
                if q_id not in results_dict:
                    results_dict[q_id] = {}
                results_dict[q_id][d_id] = score
                
            evaluator = EvaluateRetrieval(retriever=None, k_values=k_values)
            ndcg, _map, recall, _ = evaluator.evaluate(qrels_dict, results_dict, k_values)
            
            summary_rows.append({
                "Sistema": name,
                "nDCG@10": ndcg[f"NDCG@{k_num}"],
                "MAP@10": _map[f"MAP@{k_num}"],
                "Recall@10": recall[f"Recall@{k_num}"]
            })
            
    return pd.DataFrame(summary_rows)

In [17]:
index_bm25 = build_bm25_index(df_corpus["text"].tolist())

# 2. Obtener los DataFrames de cambios/resultados que ya generan tus funciones
df_cambios_ce = run_cross_encoder_reranking(
    df_queries, df_corpus, index_bm25, top_k_baseline=20, top_n_final=10
)
df_cambios_ltr = run_ltr_reranking(
    df_queries,
    df_corpus,
    index_bm25,
    ground_truth=qrels,
    top_k_baseline=20,
    top_n_final=10,
)

def df_to_beir_results(df, score_column):
    beir_results = {}
    for _, row in df.iterrows():
        q_id = str(row["Query_ID"])
        d_id = str(row["Doc_ID"])
        score = 1 / int(row["Rank_CE"] if "Rank_CE" in row else row["Rank_LTR"])

        if q_id not in beir_results:
            beir_results[q_id] = {}
        beir_results[q_id][d_id] = score
    return beir_results

baseline_dict = run_beir_evaluation(
    df_queries, df_corpus, qrels, index_bm25
)
ce_dict = df_to_beir_results(df_cambios_ce, "Rank_CE")
ltr_dict = df_to_beir_results(df_cambios_ltr, "Rank_LTR")

df_final_metricas = evaluate_pipeline_with_beir(
    qrels_dict=qrels,
    baseline_results=baseline_dict,
    ce_results=ce_dict,
    ltr_results=ltr_dict,
    k_values=[10],
)

print(df_final_metricas)

Cargando modelo Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Entrenando el modelo LTR (LambdaRANK)...


C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
C:\Users\leand\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
C:\Users\leand\AppData\Local\Packages

           Sistema  nDCG@10  MAP@10  Recall@10
0  Baseline (BM25)  0.64835     0.0    0.76717
